# Nemotron LoRA — train directly on Colab (A100 40GB, 4-bit QLoRA)

No Modal needed. We use **4-bit NF4 QLoRA with bf16 compute**: the 31.6B model quantizes to ~16 GB (fits a 40 GB A100 with room to spare) and all math stays bf16, which avoids the 8-bit MoE dtype error.

**First:** Runtime -> Change runtime type -> **A100 GPU** (Colab Pro). Run cells top to bottom; the moment training finishes, run the *Save adapter* cell (Colab can drop).

## 0. Confirm an A100 (~40 GB)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Code + dependencies

In [ ]:
!git clone -b build/nemotron-pipeline https://github.com/SebAustin/NVIDIA-Nemotron-Model-Reasoning-Challenge repo
%cd repo
!pip install -q "transformers>=4.45,<5" peft trl datasets accelerate bitsandbytes psutil hf_transfer einops

## 2. Build mamba_ssm + causal_conv1d from source (matches Colab's torch, ~5-10 min)

In [ ]:
import os, torch
print("torch", torch.__version__, "cuda", torch.version.cuda)
!pip uninstall -y -q mamba-ssm causal-conv1d
os.environ["CAUSAL_CONV1D_FORCE_BUILD"]="TRUE"; os.environ["MAMBA_FORCE_BUILD"]="TRUE"; os.environ["MAX_JOBS"]="4"
!pip install -q ninja packaging wheel setuptools
!pip install -q --no-build-isolation causal-conv1d
!pip install -q --no-build-isolation mamba-ssm
!python -c "import causal_conv1d, mamba_ssm; print('mamba OK')"

## 3. Hugging Face login (base model may be gated)

In [ ]:
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
from huggingface_hub import login
login()

## 4. Competition data (your Kaggle token)

In [ ]:
import os, urllib.request
TOK = "KGAT_xxxxxxxxxxxxxxxx"   # <-- paste your Kaggle API token
os.makedirs('data', exist_ok=True)
url = 'https://www.kaggle.com/api/v1/competitions/data/download/nvidia-nemotron-model-reasoning-challenge/train.csv'
req = urllib.request.Request(url, headers={'Authorization': f'Bearer {TOK}'})
open('data/train.csv','wb').write(urllib.request.urlopen(req).read())
print('train.csv:', os.path.getsize('data/train.csv'), 'bytes')

## 5. Build the SFT data

In [ ]:
!python scripts/01_eda.py --data-dir data
!python scripts/02_prepare_data.py --data-dir data

## 6. Train (4-bit QLoRA, single 40 GB A100)
Downloads the base model (~63 GB, one-time per session), then trains. The smoke test runs first to confirm it fits.

In [ ]:
import os
os.environ['QUANT'] = '4bit'                       # NF4 + bf16 compute (fits 40GB)
os.environ['NEMOTRON_MAX_MEMORY_GPU'] = '38GiB'
os.environ['SFT_MAX_SEQ_LENGTH'] = '1024'          # data is short
!python scripts/03_train_lora.py --data-path data/train_sft.jsonl --output-dir lora_adapter

## 7. Save the adapter OFF Colab (do this immediately)

In [ ]:
import shutil
shutil.make_archive('/content/lora_adapter', 'zip', 'lora_adapter')
from google.colab import files; files.download('/content/lora_adapter.zip')

## 8. Next: package + submit on Kaggle
Upload `lora_adapter` as a Kaggle dataset, run `kaggle_package_submit.ipynb` -> Save Version -> Submit.

*If 4-bit ever OOMs on a smaller GPU, drop `SFT_MAX_SEQ_LENGTH` to 768 and `LORA_R` to 8 (set `os.environ['LORA_R']='8'`).*